In [0]:
%pip install pystac-client planetary-computer rasterio

In [0]:
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import pystac_client
import planetary_computer
import rasterio

from pyspark.sql import Row

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

ufs = (
    spark.table("gs_carbon.metadata.uf_coordinates")
    .toPandas()
)

resultados = []

for _, row in ufs.iterrows():

    uf = row["uf"]
    latitude = float(row["latitude"])
    longitude = float(row["longitude"])

    print(f"Processando UF {uf}")

    try:

        delta = 0.5

        bbox = [
            longitude - delta,
            latitude - delta,
            longitude + delta,
            latitude + delta
        ]

        search = catalog.search(
            collections=["sentinel-2-l2a"],
            bbox=bbox,
            datetime="2025-01-01/2025-12-31",
            query={
                "eo:cloud_cover": {
                    "lt": 10
                }
            },
            limit=10
        )

        items = list(search.items())

        if len(items) == 0:
            print(f"Nenhuma imagem encontrada para {uf}")
            continue

        item = sorted(
            items,
            key=lambda x: (
                x.properties.get("eo:cloud_cover", 100),
                x.datetime
            )
        )[0]

        item = planetary_computer.sign(item)

        red_url = item.assets["B04"].href
        nir_url = item.assets["B08"].href

        with rasterio.open(red_url) as src:
            red = src.read(1)

        with rasterio.open(nir_url) as src:
            nir = src.read(1)

        red = red.astype("float32")
        nir = nir.astype("float32")

        ndvi = np.where(
            (nir + red) == 0,
            np.nan,
            (nir - red) / (nir + red)
        )

        resultados.append(
            Row(
                uf=uf,
                latitude=latitude,
                longitude=longitude,
                data_imagem=str(item.datetime),
                cloud_cover=float(
                    item.properties.get("eo:cloud_cover", 0)
                ),
                sentinel_tile=item.properties.get(
                    "s2:mgrs_tile",
                    ""
                ),
                ndvi_min=float(np.nanmin(ndvi)),
                ndvi_medio=float(np.nanmean(ndvi)),
                ndvi_max=float(np.nanmax(ndvi))
            )
        )

        print(f"{uf} concluído")

    except Exception as e:

        print(f"Erro na UF {uf}: {str(e)}")

df_resultado = spark.createDataFrame(resultados)

display(df_resultado)

df_resultado.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gs_carbon.bronze.sentinel_ndvi")

In [0]:
print(item.assets.keys())

In [0]:
print(red_url)

In [0]:
print(red.shape)
print(nir.shape)

In [0]:
import numpy as np

red = red.astype("float32")
nir = nir.astype("float32")

ndvi = np.where(
    (nir + red) == 0,
    np.nan,
    (nir - red) / (nir + red)
)

In [0]:
print("NDVI mínimo:", np.nanmin(ndvi))
print("NDVI máximo:", np.nanmax(ndvi))
print("NDVI médio:", np.nanmean(ndvi))

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))

plt.imshow(
    ndvi,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="NDVI")
plt.title("NDVI Sentinel-2")
plt.show()